# Imports

In [1]:
import os
import time
import tracemalloc
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

from xgboost import XGBClassifier

In [2]:
data = pd.read_csv("/kaggle/input/allflowmeter-hikari2021/ALLFLOWMETER_HIKARI2021.csv")

In [3]:
X = data.drop(columns=["Label", "traffic_category", "uid", "originh", "responh","Unnamed: 0.1", "Unnamed: 0"], errors="ignore")

X = X.dropna()
kept_indices = X.index

y = data.loc[kept_indices, "traffic_category"]

print(X.shape)
print(y.shape)

(555278, 81)
(555278,)


In [5]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from collections import Counter

In [6]:
print("\n===================================================================")
print("\nValue of y:")
print(y)
print("\n===================================================================")

le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)

print("\n🔹 Label Encoding Map (Index → Class Name):")
for i, cls in enumerate(le.classes_):
    print(f"{i} → {cls}")
# print(y_encoded.value_counts())

print("\nValue of y_encoded:")
print(y_encoded)

y_categorical = to_categorical(y_encoded, num_classes=num_classes)
print("\n===================================================================")
print("\nValue of y_categorical:")
print(y_categorical)
print("\n===================================================================")



Value of y:
0              Bruteforce-XML
1              Bruteforce-XML
2              Bruteforce-XML
3              Bruteforce-XML
4              Bruteforce-XML
                 ...         
555273    XMRIGCC CryptoMiner
555274    XMRIGCC CryptoMiner
555275    XMRIGCC CryptoMiner
555276    XMRIGCC CryptoMiner
555277    XMRIGCC CryptoMiner
Name: traffic_category, Length: 555278, dtype: object


🔹 Label Encoding Map (Index → Class Name):
0 → Background
1 → Benign
2 → Bruteforce
3 → Bruteforce-XML
4 → Probing
5 → XMRIGCC CryptoMiner

Value of y_encoded:
[3 3 3 ... 5 5 5]


Value of y_categorical:
[[0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 ...
 [0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 1.]]



In [7]:
print(X.shape)

(555278, 81)


# Train-Test Split

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("\nTrain/Test Split Done")
print("Train size:", X_train.shape)
print("Test size :", X_test.shape)

print(X)


Train/Test Split Done
Train size: (444222, 81)
Test size : (111056, 81)
        originp  responp  flow_duration  fwd_pkts_tot  bwd_pkts_tot  \
0         13316      443       2.207588            15            14   
1         13318      443      15.624266            15            14   
2         13320      443      12.203357            14            13   
3         13322      443       9.992448            14            13   
4         13324      443       7.780611            14            14   
...         ...      ...            ...           ...           ...   
555273      138      138       0.000000             1             0   
555274      138      138       0.000000             1             0   
555275      138      138       0.000000             1             0   
555276      138      138       0.000000             1             0   
555277      138      138       0.000000             1             0   

        fwd_data_pkts_tot  bwd_data_pkts_tot  fwd_pkts_per_sec  \
0       

# Standardization

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("\nStandard Scaling Applied")
print("Train scaled:", X_train_scaled.shape)
print("Test scaled :", X_test_scaled.shape)

print(X)


Standard Scaling Applied
Train scaled: (444222, 81)
Test scaled : (111056, 81)
        originp  responp  flow_duration  fwd_pkts_tot  bwd_pkts_tot  \
0         13316      443       2.207588            15            14   
1         13318      443      15.624266            15            14   
2         13320      443      12.203357            14            13   
3         13322      443       9.992448            14            13   
4         13324      443       7.780611            14            14   
...         ...      ...            ...           ...           ...   
555273      138      138       0.000000             1             0   
555274      138      138       0.000000             1             0   
555275      138      138       0.000000             1             0   
555276      138      138       0.000000             1             0   
555277      138      138       0.000000             1             0   

        fwd_data_pkts_tot  bwd_data_pkts_tot  fwd_pkts_per_sec  \
0

# Training Model without PCA without SMOTE


In [12]:
NUM_CLASSES = y.nunique()

model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=1.5,
    objective="multi:softprob",
    num_class=NUM_CLASSES,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

In [15]:
tracemalloc.start()
train_start_0 = time.perf_counter()

model.fit(X_train_scaled, y_train)

train_time_0 = time.perf_counter() - train_start_0
current_0, peak_ram_0 = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Training Time: {train_time_0:.2f} sec")
print(f"Peak RAM Usage: {peak_ram_0 / (1024**2):.2f} MB")

Training Time: 128.41 sec
Peak RAM Usage: 4.65 MB


In [16]:
y_pred_0 = model.predict(X_test_scaled)
y_proba_0 = model.predict_proba(X_test_scaled)

acc_0 = accuracy_score(y_test, y_pred_0)
f1_macro_0 = f1_score(y_test, y_pred_0, average="macro")
f1_weighted_0 = f1_score(y_test, y_pred_0, average="weighted")

print(f"Accuracy: {acc_0:.4f}")
print(f"F1 Macro: {f1_macro_0:.4f}")
print(f"F1 Weighted: {f1_weighted_0:.4f}")


Accuracy: 0.7753
F1 Macro: 0.5078
F1 Weighted: 0.7610


# PCA

In [17]:
from sklearn.decomposition import PCA
import numpy as np

pca_temp = PCA()
pca_temp.fit(X_train_scaled)

cumulative_variance = np.cumsum(pca_temp.explained_variance_ratio_)
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1

print(f"\nNumber of PCA components for 95% variance: {n_components_95}")

pca = PCA(n_components=n_components_95)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

print("Train PCA shape:", X_train_pca.shape)
print("Test PCA shape :", X_test_pca.shape)



Number of PCA components for 95% variance: 27
Train PCA shape: (444222, 27)
Test PCA shape : (111056, 27)


# With PCA Without SMOTE

In [17]:
tracemalloc.start()
train_start_1 = time.perf_counter()

model.fit(X_train_pca, y_train)

train_time_1 = time.perf_counter() - train_start_1
current_1, peak_ram_1 = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Training Time: {train_time_1:.2f} sec")
print(f"Peak RAM Usage: {peak_ram_1 / (1024**2):.2f} MB")

Training Time: 71.20 sec
Peak RAM Usage: 4.24 MB


In [18]:
y_pred_1 = model.predict(X_test_pca)
y_proba_1 = model.predict_proba(X_test_pca)

acc_1 = accuracy_score(y_test, y_pred_1)
f1_macro_1 = f1_score(y_test, y_pred_1, average="macro")
f1_weighted_1 = f1_score(y_test, y_pred_1, average="weighted")

print(f"Accuracy: {acc_1:.4f}")
print(f"F1 Macro: {f1_macro_1:.4f}")
print(f"F1 Weighted: {f1_weighted_1:.4f}")


Accuracy: 0.7548
F1 Macro: 0.5200
F1 Weighted: 0.7386


# SMOTE


In [18]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)

X_train_final, y_train_final = smote.fit_resample(
    X_train_pca, y_train
)

print("\nSMOTE Applied on PCA-transformed TRAIN data")
print("X_train_final:", X_train_final.shape)
print("Class distribution after SMOTE:", Counter(y_train_final))



SMOTE Applied on PCA-transformed TRAIN data
X_train_final: (1667670, 27)
Class distribution after SMOTE: Counter({np.int64(0): 277945, np.int64(4): 277945, np.int64(1): 277945, np.int64(2): 277945, np.int64(3): 277945, np.int64(5): 277945})


In [19]:
print("\nFINAL DATASETS READY")
print("Train features:", X_train_final.shape)
print("Train labels  :", y_train_final.shape)
print("Test features :", X_test_pca.shape)
print("Test labels   :", y_test.shape)


FINAL DATASETS READY
Train features: (1667670, 27)
Train labels  : (1667670,)
Test features : (111056, 27)
Test labels   : (111056,)


# Without PCA and with SMOTE

In [13]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)

X_train_2, y_train_2 = smote.fit_resample(
    X_train_scaled, y_train
)

print("\nSMOTE Applied on PCA-transformed TRAIN data")
print("X_train_final:", X_train_2.shape)
print("Class distribution after SMOTE:", Counter(y_train_2))



SMOTE Applied on PCA-transformed TRAIN data
X_train_final: (1667670, 81)
Class distribution after SMOTE: Counter({np.int64(0): 277945, np.int64(4): 277945, np.int64(1): 277945, np.int64(2): 277945, np.int64(3): 277945, np.int64(5): 277945})


In [14]:
print("\nFINAL DATASETS READY")
print("Train features:", X_train_2.shape)
print("Train labels  :", y_train_2.shape)
print("Test features :", X_test_scaled.shape)
print("Test labels   :", y_test.shape)


FINAL DATASETS READY
Train features: (1667670, 81)
Train labels  : (1667670,)
Test features : (111056, 81)
Test labels   : (111056,)


In [15]:
tracemalloc.start()
train_start_2 = time.perf_counter()

model.fit(X_train_2, y_train_2)

train_time_2 = time.perf_counter() - train_start_2
current_2, peak_ram_2 = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Training Time: {train_time_2:.2f} sec")
print(f"Peak RAM Usage: {peak_ram_2 / (1024**2):.2f} MB")

Training Time: 543.51 sec
Peak RAM Usage: 15.91 MB


In [16]:
y_pred_2 = model.predict(X_test_scaled)
y_proba_2 = model.predict_proba(X_test_scaled)

acc_2 = accuracy_score(y_test, y_pred_2)
f1_macro_2 = f1_score(y_test, y_pred_2, average="macro")
f1_weighted_2 = f1_score(y_test, y_pred_2, average="weighted")

print(f"Accuracy: {acc_2:.4f}")
print(f"F1 Macro: {f1_macro_2:.4f}")
print(f"F1 Weighted: {f1_weighted_2:.4f}")


Accuracy: 0.7453
F1 Macro: 0.6548
F1 Weighted: 0.7526


# Model configuration

In [26]:
NUM_CLASSES = y.nunique()

model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=1.5,
    objective="multi:softprob",
    num_class=NUM_CLASSES,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

# With PCA and with SMOTE

In [27]:
tracemalloc.start()
train_start = time.perf_counter()

model.fit(X_train_final, y_train_final)

train_time = time.perf_counter() - train_start
current, peak_ram = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Training Time: {train_time:.2f} sec")
print(f"Peak RAM Usage: {peak_ram / (1024**2):.2f} MB")

Training Time: 234.80 sec
Peak RAM Usage: 15.91 MB


# Inference timing (batch=1 & small batch)

In [35]:
def measure_latency(model, X, batch_size=1, runs=100):
    times = []
    for _ in range(runs):
        idx = np.random.choice(len(X), batch_size, replace=False)
        start = time.perf_counter()
        model.predict(X[idx])
        times.append(time.perf_counter() - start)

    times = np.array(times)

    # print(times)
    return np.percentile(times, [50, 95]), batch_size / times.mean()


In [ ]:
lat_1, thr_1 = measure_latency(model, X_test_pca, batch_size=1)
lat_32, thr_32 = measure_latency(model, X_test_pca, batch_size=32)

print(f"Latency batch=1 (p50/p95): {lat_1}")
print(f"Latency batch=32 (p50/p95): {lat_32}")
print(f"Throughput batch=1: {thr_1:.2f} samples/s")
print(f"Throughput batch=32: {thr_32:.2f} samples/s")


In [36]:
lat_1, thr_1 = measure_latency(model, X_test_pca, batch_size=1)
lat_32, thr_32 = measure_latency(model, X_test_pca, batch_size=32)

print(f"Latency batch=1 (p50/p95): {lat_1}")
print(f"Latency batch=32 (p50/p95): {lat_32}")
print(f"Throughput batch=1: {thr_1:.2f} samples/s")
print(f"Throughput batch=32: {thr_32:.2f} samples/s")


[0.00358105 0.00154426 0.00116798 0.00121158 0.00116277 0.00120589
 0.00113298 0.00104959 0.00124998 0.00112094 0.00105073 0.00103835
 0.00102692 0.00107134 0.00103386 0.00104997 0.00102865 0.00106962
 0.00103116 0.0010725  0.00103841 0.00102309 0.00103441 0.001027
 0.00102687 0.00111964 0.00102367 0.00101561 0.00105452 0.00101352
 0.00103452 0.00101815 0.00102344 0.00102686 0.00103523 0.00137106
 0.00102144 0.00103478 0.00102023 0.00106627 0.00109367 0.0010154
 0.00105444 0.00111322 0.00102503 0.00103577 0.00102153 0.00103307
 0.00102848 0.00104905 0.00110896 0.00103535 0.00106744 0.00106032
 0.00103518 0.00102666 0.00102296 0.00100987 0.00102162 0.00101489
 0.00102785 0.00107325 0.00101908 0.0009994  0.00099163 0.00099377
 0.00097265 0.00103273 0.00097319 0.00103009 0.00108382 0.00099327
 0.00181511 0.00099267 0.00098422 0.00097776 0.00098339 0.00105983
 0.00097973 0.00098168 0.00097094 0.00118021 0.00100825 0.00107538
 0.00098757 0.00112714 0.00103899 0.00098065 0.00105489 0.0013634

# Predictions

In [37]:
y_pred = model.predict(X_test_pca)
y_proba = model.predict_proba(X_test_pca)

# Core classification metrics

In [38]:
acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print(f"Accuracy: {acc:.4f}")
print(f"F1 Macro: {f1_macro:.4f}")
print(f"F1 Weighted: {f1_weighted:.4f}")


Accuracy: 0.6802
F1 Macro: 0.6395
F1 Weighted: 0.6920


# AUROC & AUPRC (imbalance-aware)

In [35]:
auroc = roc_auc_score(y_test, y_proba, multi_class="ovr")

auprc = average_precision_score(
    pd.get_dummies(y_test),
    y_proba,
    average="macro"
)

print(f"AUROC (OvR): {auroc:.4f}")
print(f"AUPRC (Macro): {auprc:.4f}")


AUROC (OvR): 0.9336
AUPRC (Macro): 0.5921


# Calibration (Brier score)

In [36]:
brier = np.mean([
    brier_score_loss((y_test == i).astype(int), y_proba[:, i])
    for i in range(NUM_CLASSES)
])

print(f"Brier Score (Calibration Error): {brier:.4f}")


Brier Score (Calibration Error): 0.0774


# Confusion Matrix & report

In [37]:
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)



Classification Report:

              precision    recall  f1-score   support

           0       0.64      0.85      0.73     34030
           1       0.90      0.55      0.68     69486
           2       0.43      0.99      0.60      1177
           3       0.51      1.00      0.67      1029
           4       0.27      1.00      0.43      4678
           5       0.51      1.00      0.68       656

    accuracy                           0.67    111056
   macro avg       0.54      0.90      0.63    111056
weighted avg       0.78      0.67      0.68    111056

Confusion Matrix:
 [[28860  4388   234     2     3   543]
 [16491 38203  1311  1006 12395    80]
 [    9     4  1163     0     1     0]
 [    0     0     0  1029     0     0]
 [    1     9     0     0  4668     0]
 [    2     0     0     0     0   654]]


# Model size & params

In [38]:
model_path = "xgb_model.json"
model.save_model(model_path)

model_size_mb = os.path.getsize(model_path) / (1024**2)
num_trees = model.get_booster().num_boosted_rounds()

print(f"Model size: {model_size_mb:.2f} MB")
print(f"Number of trees: {num_trees}")


Model size: 5.14 MB
Number of trees: 300


# FLOPs / MACs (ESTIMATED)

In [39]:
avg_depth = model.max_depth
flops_per_tree = avg_depth * 2        # approx comparisons
total_flops = flops_per_tree * num_trees

print(f"Estimated FLOPs per inference: ~{total_flops}")

Estimated FLOPs per inference: ~3000


# Energy & Power (ESTIMATED – professional note)

In [40]:
# Requires hardware profiler in real systems (e.g., RAPL / nvidia-smi)
# Here we provide a conservative estimate

avg_power_watts = 45     # typical laptop CPU
energy_per_inference = avg_power_watts * lat_1[0]

train_energy_kj = avg_power_watts * train_time / 1000

print(f"Estimated Energy / Inference: {energy_per_inference:.4f} J")
print(f"Estimated Training Energy: {train_energy_kj:.2f} kJ")
print(f"Estimated Avg Power Draw: {avg_power_watts} W")


Estimated Energy / Inference: 0.0746 J
Estimated Training Energy: 7.41 kJ
Estimated Avg Power Draw: 45 W
